# 🏁 07 Training Final Optimized Model
Train the champion model with the best discovered configurations using **ResNet50**.

### 🎯 Goals:
1. Load optimal parameters from tuning logs
2. Train with ResNet50 and Optimized Mastery Head
3. Save final production model to `/models/final_optimized_model.h5`

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

import tensorflow as tf
import pandas as pd
from tensorflow.keras import layers, models, applications

BASE_PATH = '/content/drive/MyDrive/DL/'
LOG_PATH = os.path.join(BASE_PATH, 'logs/')
MODEL_PATH = os.path.join(BASE_PATH, 'models/')

# 1. Select Best Config from Experiments
results_df = pd.read_csv(os.path.join(LOG_PATH, 'hyperparameter_results.csv'))
best_config = results_df.loc[results_df['val_accuracy'].idxmax()]
print(f"⭐ Selected Best Config: {best_config.to_dict()}")

# 2. Re-Build Best Model Architecture (ResNet50 Standard)
def build_final_model(dr):
    base = applications.ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base.trainable = True
    
    model = models.Sequential([
        layers.Lambda(lambda x: applications.resnet50.preprocess_input(x)),
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dr),
        layers.Dense(7, activation='softmax')
    ])
    return model

model = build_final_model(best_config['dr'])

# 3. Final Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_config['lr']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Final Training Run
print("🚀 Starting final training with ResNet50 backbone...")
history = model.fit(
    train_ds, 
    epochs=30, 
    validation_data=val_ds,
    callbacks=[tf.keras.callbacks.ModelCheckpoint(os.path.join(MODEL_PATH, 'final_optimized_model.h5'), save_best_only=True)]
)

print(f"✅ Training Complete. Model saved to {MODEL_PATH}final_optimized_model.h5")